# Linear regression — from scratch, then with PyTorch and sklearn

Companion notebook for **Eps 19–23** of the Linear Regression series.

*One dataset (7 cars, weight → MPG). One model (`y = wx + b`). One loss (MSE). One algorithm (gradient descent). By the end you'll have trained it three ways: by hand, in PyTorch, and in sklearn.*

## 1. Setup

Just the imports. NumPy for the math, matplotlib for plots. Torch and sklearn show up later.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

## 2. The dataset

Seven cars. For each, its weight (in thousands of pounds) and how many miles per gallon it gets. Heavier cars get worse mileage — you can see it in the numbers.

In [ ]:
weights = np.array([2.4, 2.8, 3.0, 3.2, 3.6, 4.0, 4.4])
mpgs    = np.array([24,  22,  20,  19,  17,  15,  14])

for w, m in zip(weights, mpgs):
    print(f'  {w}K lbs  ->  {m} MPG')

### Plot it

A scatter is the right first move. Always plot the data before you model it.

In [ ]:
plt.scatter(weights, mpgs, s=80, color='#00E5FF', edgecolor='black', linewidth=1.5, zorder=3)
plt.xlabel('weight (1000 lbs)')
plt.ylabel('MPG')
plt.title('7 cars — weight vs MPG')
plt.show()

## 3. The model: `y = w·x + b`

Simplest possible model for this data is a straight line. Two parameters — `w` (slope) and `b` (intercept). Pick any (w, b), get a different line.

In [ ]:
def predict(w, b, x):
    return w * x + b

# Wild guess: w = -3, b = 30
w_guess, b_guess = -3.0, 30.0

# Predict MPG for each car
preds = predict(w_guess, b_guess, weights)

for w, m, p in zip(weights, mpgs, preds):
    print(f'  weight={w}  actual MPG={m}  predicted MPG={p:.1f}')

### Plot the guess

Plot the line over the data. It's not great. That's the point — let's measure exactly *how* not-great.

In [ ]:
xs = np.linspace(2, 5, 50)
ys = predict(w_guess, b_guess, xs)

plt.scatter(weights, mpgs, s=80, color='#00E5FF', edgecolor='black', linewidth=1.5, zorder=3, label='data')
plt.plot(xs, ys, color='#FFD27F', linewidth=2.5, label=f'guess: y = {w_guess}x + {b_guess}')
plt.xlabel('weight (1000 lbs)')
plt.ylabel('MPG')
plt.legend()
plt.show()

## 4. Loss — measuring 'wrong'

**Mean Squared Error.** For each car, square the difference between the prediction and the truth. Average across all cars. That single number says how wrong the model is.

We square because squaring gives clean gradients (Ep 20 of the calculus series — derivative of x² is 2x, smooth everywhere).

In [ ]:
def mse(w, b):
    preds = predict(w, b, weights)
    return np.mean((mpgs - preds) ** 2)

print(f'MSE of our guess (w={w_guess}, b={b_guess}): {mse(w_guess, b_guess):.3f}')

### Visualize the errors — the gray squares

MSE has a picture. The squared error for each car is the area of a square whose side is the vertical distance from the line to the point. Total area = sum of squared errors.

In [ ]:
fig, ax = plt.subplots()
ax.scatter(weights, mpgs, s=80, color='#00E5FF', edgecolor='black', linewidth=1.5, zorder=5, label='data')
ax.plot(xs, predict(w_guess, b_guess, xs), color='#FFD27F', linewidth=2.5, label='guess')

# Squared-error squares, drawn from each point to the line
for w_data, m_data in zip(weights, mpgs):
    p = predict(w_guess, b_guess, w_data)
    err = m_data - p
    # Draw a square whose side is |err|
    side = abs(err)
    ax.add_patch(plt.Rectangle((w_data, min(m_data, p)), side, side,
                                color='#7C4DFF', alpha=0.25, linewidth=0))
    ax.plot([w_data, w_data], [m_data, p], color='#7C4DFF', alpha=0.6, linewidth=1)

ax.set_xlabel('weight (1000 lbs)')
ax.set_ylabel('MPG')
ax.set_title('The squared distances — total area = MSE × N')
ax.set_aspect('equal')
ax.legend()
plt.show()

### Try a better guess

Drag your mental slider — change w and b until MSE is smaller. The lower the MSE, the better the fit.

In [ ]:
w_better, b_better = -5.0, 34.0
print(f'Old guess MSE: {mse(w_guess, b_guess):.3f}')
print(f'New guess MSE: {mse(w_better, b_better):.3f}')

## 5. The loss surface

MSE is a function of (w, b). Two inputs, one output. That's a surface. Let's compute it on a grid and plot it.

*Reusing the bowl from calculus Ep 17 — but this bowl is built from the actual seven cars in our dataset.*

In [ ]:
w_range = np.linspace(-10, 2, 60)
b_range = np.linspace(15, 45, 60)
W, B = np.meshgrid(w_range, b_range)

# Vectorized MSE computation over the grid
loss_grid = np.zeros_like(W)
for i in range(W.shape[0]):
    for j in range(W.shape[1]):
        loss_grid[i, j] = mse(W[i, j], B[i, j])

print('grid shape:', loss_grid.shape, ' min MSE on grid:', loss_grid.min().round(3))

### Plot the bowl

Tilt the 3D plot and look at it. One minimum. No local pits. That's why linear regression always trains.

In [ ]:
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(W, B, loss_grid, cmap='viridis', alpha=0.8, edgecolor='none')
ax.contour(W, B, loss_grid, zdir='z', offset=loss_grid.min(),
           cmap='viridis', alpha=0.5)
ax.set_xlabel('w (slope)')
ax.set_ylabel('b (intercept)')
ax.set_zlabel('MSE')
ax.set_title('The loss surface — a clean convex bowl')
ax.view_init(elev=30, azim=-60)
plt.show()

## 6. Gradient descent by hand

Compute the gradient (the slope of the bowl at the current point). Take a step downhill. Repeat. Same algorithm from calculus Ep 17.

For MSE with `y = wx + b`, the gradients work out to:

$$\frac{\partial L}{\partial w} = \frac{2}{N}\sum_i x_i (wx_i + b - y_i)$$

$$\frac{\partial L}{\partial b} = \frac{2}{N}\sum_i (wx_i + b - y_i)$$

In [ ]:
def grad(w, b):
    errs = predict(w, b, weights) - mpgs
    dw = 2 * np.mean(weights * errs)
    db = 2 * np.mean(errs)
    return dw, db

# At our better guess, what does the gradient look like?
dw, db = grad(w_better, b_better)
print(f'∂L/∂w = {dw:.3f}    ∂L/∂b = {db:.3f}')

### One step

In [ ]:
lr = 0.01
w_new = w_better - lr * dw
b_new = b_better - lr * db

print(f'before:  w={w_better:.3f}  b={b_better:.3f}  MSE={mse(w_better, b_better):.3f}')
print(f'after:   w={w_new:.3f}  b={b_new:.3f}  MSE={mse(w_new, b_new):.3f}')

### The full loop

Start somewhere random. Step downhill. Repeat until the gradients are small.

In [ ]:
w, b = -1.0, 15.0   # starting point — wrong on purpose so the descent is visible
lr = 0.01           # small enough that w doesn't oscillate
n_steps = 10000     # ten thousand steps. takes about a second.

history = {'w': [w], 'b': [b], 'mse': [mse(w, b)]}

for step in range(n_steps):
    dw, db = grad(w, b)
    w -= lr * dw
    b -= lr * db
    history['w'].append(w)
    history['b'].append(b)
    history['mse'].append(mse(w, b))

print(f'final:  w={w:.4f}  b={b:.4f}  MSE={mse(w, b):.4f}')
print(f'(true optimum is roughly w=-5.15, b=35.92 — see sklearn below)')
print()
print('Notice: this takes 10,000 iterations to converge with vanilla SGD on raw features.')
print('Real ML normalizes features first OR uses a smarter optimizer (Adam). Both fix this.')

### Plot the descent path on the bowl

In [ ]:
fig = plt.figure(figsize=(11, 7))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(W, B, loss_grid, cmap='viridis', alpha=0.55, edgecolor='none')

# The path
ax.plot(history['w'], history['b'], history['mse'],
        color='#FFD27F', linewidth=2, marker='o', markersize=3)

# Start + end markers
ax.scatter([history['w'][0]], [history['b'][0]], [history['mse'][0]],
           color='#FF4081', s=150, edgecolor='white', linewidth=1.5, label='start')
ax.scatter([history['w'][-1]], [history['b'][-1]], [history['mse'][-1]],
           color='#00E676', s=150, edgecolor='white', linewidth=1.5, label='end')

ax.set_xlabel('w'); ax.set_ylabel('b'); ax.set_zlabel('MSE')
ax.set_title('Gradient descent on the loss surface')
ax.view_init(elev=30, azim=-60)
ax.legend()
plt.show()

### Plot the fitted line over the data

In [ ]:
plt.scatter(weights, mpgs, s=80, color='#00E5FF', edgecolor='black', linewidth=1.5, zorder=3, label='data')
plt.plot(xs, predict(w, b, xs), color='#FFD27F', linewidth=2.5,
         label=f'fitted: y = {w:.2f}x + {b:.2f}')
plt.xlabel('weight (1000 lbs)')
plt.ylabel('MPG')
plt.title('Linear regression — fitted by hand')
plt.legend()
plt.show()

## 7. Same thing in PyTorch — the 5-line training loop

The same gradient descent, but `loss.backward()` computes gradients for us. This is the same five lines from calculus Ep 18.

In [ ]:
import torch
import torch.nn as nn

# Convert data to tensors. PyTorch wants shape (N, 1) for the linear layer.
X = torch.tensor(weights, dtype=torch.float32).view(-1, 1)
y = torch.tensor(mpgs,    dtype=torch.float32).view(-1, 1)

# The model: y = wx + b. nn.Linear with 1 input feature, 1 output.
model = nn.Linear(1, 1)
# Adam adapts the learning rate per parameter — much more forgiving than vanilla SGD
# when the features aren't normalized (which they aren't here).
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
loss_fn = nn.MSELoss()

# THE 5-LINE TRAINING LOOP
for step in range(5000):
    pred = model(X)
    loss = loss_fn(pred, y)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

w_torch = model.weight.item()
b_torch = model.bias.item()
print(f'PyTorch result:  w={w_torch:.4f}  b={b_torch:.4f}')
print(f'By-hand result:  w={w:.4f}  b={b:.4f}')

## 8. Same thing in sklearn — 3 lines

Sklearn wraps the whole loop. For most projects, this is what you'd actually write.

In [ ]:
from sklearn.linear_model import LinearRegression

model_sk = LinearRegression()
model_sk.fit(weights.reshape(-1, 1), mpgs)

print(f'sklearn result:  w={model_sk.coef_[0]:.4f}  b={model_sk.intercept_:.4f}')
print(f'PyTorch result:  w={w_torch:.4f}  b={b_torch:.4f}')
print(f'By-hand result:  w={w:.4f}  b={b:.4f}')

# Predict MPG for a 3,500-pound car
new_car_mpg = model_sk.predict([[3.5]])
print(f'\nA 3,500-pound car -> {new_car_mpg[0]:.1f} MPG')

## 9. More features

Real cars have more than one feature. Let's pretend our cars also have horsepower and engine displacement.

In [ ]:
# Synthetic horsepower + displacement (correlated with weight, plus noise)
horsepower   = weights * 60 + np.random.normal(0, 15, size=7)
displacement = weights * 1.5 + np.random.normal(0, 0.4, size=7)

X_multi = np.column_stack([weights, horsepower, displacement])
print('shape of X_multi:', X_multi.shape)
print(X_multi.round(2))

In [ ]:
model_multi = LinearRegression()
model_multi.fit(X_multi, mpgs)

print(f'  weight coef:        {model_multi.coef_[0]:+.4f}')
print(f'  horsepower coef:    {model_multi.coef_[1]:+.4f}')
print(f'  displacement coef:  {model_multi.coef_[2]:+.4f}')
print(f'  intercept:          {model_multi.intercept_:+.4f}')

# Each weight tells you: 'holding the others constant, this is how much MPG
# changes per unit of this feature.'

## 10. What we did

- Loaded a dataset and plotted it.
- Defined a model `y = wx + b` and a loss `MSE`.
- Visualized the loss as a surface (the bowl).
- Trained the model three ways: by-hand gradient descent, the PyTorch 5-line loop, and sklearn's 3 lines.
- All three give the same answer (within numerical precision).
- Extended to multiple features — one extra weight per feature, same algorithm.

**Next notebook: logistic regression.** Same training loop, two substitutions — sigmoid in the forward pass, log loss in place of MSE.